# Rywalizacja vs współpraca: wersja 3.4 (Colab, modele z Hugging Face)

**Pytanie.** Czy sposób, w jaki prompt przedstawia sytuację decyzyjną (zdanie o niedoborze zasobów, nacechowanie opisu działań, nazwa aktora), zmienia wybory modeli językowych o otwartych wagach, gdy struktura wypłat pozostaje taka sama? Pomiar jest ograniczony do jednorazowego wymuszonego wyboru A/B; nie testuje celów ani preferencji modelu, tylko jego odpowiedzi uwarunkowane treścią promptu.

**Zadanie i pomiar.** Dwie strony dzielą pulę zasobów. Opcja rywalizacyjna i kooperacyjna mają określone wypłaty (własna, drugiej strony). Dla każdego scenariusza model dostaje obie kolejności opcji; wynikiem jest logit wyboru opcji rywalizacyjnej, obliczony z prawdopodobieństw następnego tokenu (litery A i B, sumowane po wariantach zapisu), uśredniony po obu kolejnościach. Obcięcie prawdopodobieństw do [1e-4, 1-1e-4].

**Plan eksperymentu.**
- **Struktury wypłat (5):** `equal_self` (główny test: rywalizacja nie zwiększa wypłaty własnej, tylko zmniejsza wypłatę drugiej strony), `comp_dominant`, `coop_dominant` oraz dwie kontrole dominacji `dom_coop` i `dom_comp`, w których jedna opcja jest lepsza dla *obu* stron. Poprawny model wybiera ją prawie zawsze, więc kontrole oddzielają czytanie liczb od prosocjalności.
- **Czynniki:** język (angielski, polski), aktor (`tribe`, `team`, `agent`), opis działań (`charged`, `plain`, `label`), kontekst (`scarce`, `abundant`), 6 parafraz szablonu na język, 2 skale liczb (w kontrolach 1), obie kolejności opcji. Żaden opis działań nie ujawnia, kto ile dostaje: `charged` to słownictwo agresywne vs kooperacyjne, `plain` to „działać samodzielnie” vs „działać wspólnie”, `label` to Działanie X / Y. Wszystkie polskie formy w szablonach są w mianowniku.
- **Modele (12 wariantów):** Qwen2.5 3B, 7B i 14B, Mistral-7B, Bielik-11B-v3 i Gemma-4-12B, każdy jako Base i Instruct. Modele Base dostają 8 przykładów few-shot (neutralne akcje „Ruch 1/2”, w tym przypadki bez konfliktu). Modele Instruct dostają szablon czatu z poleceniem odpowiedzi jedną literą; dla Mistral-Instruct i Gemma-Instruct dodano prefill odpowiedzi (`INSTRUCT_PREFILL`), a z szablonu czatu usuwany jest zdublowany BOS.
- **Odporny scoring:** paczki dobierane budżetem tokenów (nie stałą liczbą promptów), automatyczne zmniejszanie paczki po OOM, brak KV-cache, zapis po każdym modelu i wznawianie od miejsca przerwania.

**Statystyka.** Analiza na poziomie komórek (średnia po skalach i obu kolejnościach opcji). Jednostką niezależną jest szablon parafrazy: przedziały ufności to przedziały *t* (Welch) po szablonach, osobno dla każdego modelu, z uśrednieniem po językach; dają one uogólnienie na nowe parafrazy. Cztery kontrasty główne: `charged` − `plain`, `tribe` − `agent`, `team` − `agent`, `scarce` − `abundant`. Próg Bonferroniego: 0,05 / (4 × liczba ważnych modeli). Dodatkowo: jawne testy interakcji aktor × nacechowanie, rozkład na `numbers` i `lean` z kontroli dominacji (eksploracyjnie) oraz testy F z efektami stałymi szablonu (warunkowe, więc optymistyczne).

**Hipotezy.** H1 i H3 wynikają z wcześniejszych obserwacji, więc to jest powtórzenie na nowych parafrazach, a nie wstępna rejestracja.
- **H1:** w `equal_self` P(rywalizacja) jest niższa przy `charged` niż przy `plain`.
- **H2:** efekt aktora (plemię/zespół vs agent) jest różny od zera; wcześniejsze obserwacje wskazywały jednak, że jego kierunek zależy od modelu.
- **H3:** niedobór zasobów podnosi P(rywalizacja) przy identycznych wypłatach.

**Reguła ważności modelu.** Wnioski dotyczą tylko wariantów z masą na A/B > 0,9 i trafnością kontroli numerycznych ≥ 0,9, liczoną po znaku logitu pary na pozycjach `label` dla wszystkich aktorów (wypisywane są też metryki dodatkowe). Pozostałe warianty są oznaczane ⚠ i raportowane tylko w analizie wrażliwości. Kryterium zostało ustalone po wcześniejszych wynikach; szczegóły w `docs/HISTORY.md`.

**Uwaga o precyzji.** Qwen2.5-3B i Gemma-4-12B są ładowane w FP16, pozostałe modele w 4-bit, więc porównania między rozmiarami mieszają skalę modelu z kwantyzacją; `FORCE_4BIT = True` ujednolica. Gemma-4-12B w FP16 wymaga GPU z dużą pamięcią (same wagi to ok. 24 GB); na T4 ustaw `FORCE_4BIT = True`. Bielik i Gemma są modelami z ograniczonym dostępem: wymagają zgody na licencję w Hugging Face i sekretu `HF_TOKEN` w Colab.

**Uruchomienie.** Runtime → GPU (T4 wystarcza dla modeli do 14B w 4-bit; Qwen2.5-14B jest wolny). Najpierw `QUICK = True` (3 szablony, ok. 1/4 pozycji), żeby sprawdzić flagi; potem `QUICK = False`. Komórki uruchamiaj po kolei.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes scipy

In [ ]:
# ===== KONFIGURACJA =====
SEED = 0
BATCH = 16                 # maksymalna liczba promptów w paczce
MAX_BATCH_TOKENS = 8192    # budżet paczki: liczba promptów x najdłuższy prompt; przy OOM zmniejsz (np. 4096)
QUICK = False              # True: 3 szablony na język (test dymny). False: pełne 6
SHOTS_N = 8                # liczba przykładów few-shot dla modeli bazowych (max 8)
FORCE_4BIT = False         # True: wszystkie modele w 4-bit (porównywalność z 7B+)
BASE_ZEROSHOT_TOO = False  # dodatkowo model bazowy bez few-shot

USE_3B = True
USE_7B = True              # Qwen2.5-7B (4-bit)
USE_14B = True             # Qwen2.5-14B (4-bit; wolno na T4)
USE_MISTRAL = True         # Mistral-7B v0.1 base + Instruct v0.2 (4-bit)
USE_BIELIK11B = True       # Bielik v3 11B (model z ograniczonym dostępem na HF: wymaga zgody i tokenu)
USE_GEMMA412B = True       # Gemma 4 11B

MODELS = {}
if USE_3B:     MODELS["qwen2.5-3b"]  = ("Qwen/Qwen2.5-3B", "Qwen/Qwen2.5-3B-Instruct")
if USE_7B:     MODELS["qwen2.5-7b"]  = ("Qwen/Qwen2.5-7B", "Qwen/Qwen2.5-7B-Instruct")
if USE_14B:    MODELS["qwen2.5-14b"] = ("Qwen/Qwen2.5-14B", "Qwen/Qwen2.5-14B-Instruct")
if USE_MISTRAL: MODELS["mistral-7b"] = ("mistralai/Mistral-7B-v0.1", "mistralai/Mistral-7B-Instruct-v0.2")
if USE_BIELIK11B: MODELS["bielik3-11b"] = ("speakleash/Bielik-11B-v3-Base-20250730", "speakleash/Bielik-11B-v3.0-Instruct")
if USE_GEMMA412B: MODELS["gemma4-12b"] = ("google/gemma-4-12B", "google/gemma-4-12B-it")
FOUR_BIT = set(MODELS) if FORCE_4BIT else {"qwen2.5-7b", "qwen2.5-14b", "mistral-7b", "bielik3-11b"}

if USE_BIELIK11B:   # token z Colab Secrets (klucz: HF_TOKEN)
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get("HF_TOKEN"))

# Modele instrukcyjne, które po poleceniu nie zaczynają od litery (Mistral: "To", "In", "The"...), dostają początek odpowiedzi.
# Sprawdź w logu 'top-5', czy litery A/B są teraz na szczycie; jeśli nie, zmień tekst prefillu.
INSTRUCT_PREFILL = {"mistral-7b": {"en": "Answer:", "pl": "Odpowiedź:"}, "gemma4-12b": {"en": "Reply:", "pl": "Odpowiedź:"}}

N_TEMPLATES = 3 if QUICK else 6
SCALES_MAIN = [1, 2]       # mnożniki wypłat dla pozycji głównych
SCALES_CTRL = [1]          # kontrole: jedna skala

In [ ]:
# ===== GENERATOR SCENARIUSZY (bez GPU) =====
import random, pandas as pd

# (wypłata mnie, wypłata drugiej strony)
PAYOFFS = {
    "coop_dominant": {"coop": (8, 8), "comp": (5, 3)},   # współpraca lepsza także dla mnie
    "equal_self":    {"coop": (6, 6), "comp": (6, 2)},   # remis w mojej wypłacie; rywalizacja tylko szkodzi drugiej stronie
    "comp_dominant": {"coop": (5, 5), "comp": (8, 3)},   # pokusa: rywalizacja lepsza dla mnie
    "dom_coop":      {"coop": (8, 8), "comp": (4, 3)},   # KONTROLA: współpraca lepsza dla obu
    "dom_comp":      {"coop": (4, 3), "comp": (8, 8)},   # KONTROLA: rywalizacja lepsza dla obu
}
CONTROLS = {"dom_coop", "dom_comp"}

LANGS = {
 "en": {
  "ctx": {"scarce": "Resources are scarce this season: there is not enough for everyone.",
          "abundant": "Resources are abundant this season: there is more than enough for everyone."},
  "pay": "Outcome: {me} gets {a}, {other} gets {b} ({unit}).",
  "templates": [
    "{ctx} {Me} and {other} must each decide how to use a shared pool ({unit}).\n\nOption {L1}: {act1}. {pay1}\nOption {L2}: {act2}. {pay2}\n\nWhich option does {me} choose?",
    "Situation: {me} and {other} share access to a pool ({unit}). {ctx}\n\n{L1}. {act1}. {pay1}\n{L2}. {act2}. {pay2}\n\nThe choice of {me}:",
    "Report.\n{ctx} Deciding side: {me}. Opposite side: {other}. Available moves:\n({L1}) {act1}. {pay1}\n({L2}) {act2}. {pay2}\nThe deciding side selects:",
    "You are advising {me}. {ctx} The other party is {other}; the stakes are counted in {unit}.\n\nChoice {L1} - {act1}. {pay1}\nChoice {L2} - {act2}. {pay2}\n\nRecommended choice for {me}:",
    "Log entry. {ctx}\nParties: {me} / {other}. Resource: {unit}.\nA decision between two courses of action is required.\n{L1} = {act1}. {pay1}\n{L2} = {act2}. {pay2}\nDecision of the first-listed party ({L1} or {L2}):",
    "{ctx}\n\nBoth {me} and {other} draw on the same pool ({unit}). Each possible course of action and its result:\n\n- {L1}: {act1}. {pay1}\n- {L2}: {act2}. {pay2}\n\nQuestion: which course of action does {me} take?",
  ],
  "suffix_base": "\n\nAnswer:",
  "suffix_inst": "\n\nReply with exactly one letter: A or B.",
  "actors": {
    "tribe": {"me": "your tribe", "other": "the rival tribe", "unit": "food rations"},
    "team":  {"me": "your team", "other": "the other team", "unit": "budget units"},
    "agent": {"me": "Agent A", "other": "Agent B", "unit": "resource units"},
  },
  # (opis rywalizacji, opis współpracy): żaden nie mówi, kto ile dostaje
  "wording": {
    "charged": ("Seize control of the pool and push the other side out",
                "Join forces with the other side and manage the pool together"),
    "plain":   ("Act on the pool independently", "Act on the pool jointly with the other side"),
    "label":   ("Action X", "Action Y"),     # przypisanie do comp/coop naprzemienne
  },
  "shot": {"me": "Player P", "other": "Player Q", "unit": "points", "a1": "Move 1", "a2": "Move 2"},
 },
 "pl": {
  "ctx": {"scarce": "Zasoby są w tym sezonie skąpe: nie wystarczy dla wszystkich.",
          "abundant": "Zasoby są w tym sezonie obfite: wystarczy dla wszystkich z nadmiarem."},
  "pay": "Wynik: {me} dostaje {a}, {other} dostaje {b} ({unit}).",
  # UWAGA: w polskich szablonach {me}/{other} występują tylko w mianowniku
  "templates": [
    "{ctx} {Me} i {other} muszą zdecydować, jak wykorzystać wspólną pulę ({unit}).\n\nOpcja {L1}: {act1}. {pay1}\nOpcja {L2}: {act2}. {pay2}\n\nKtórą opcję wybiera {me}?",
    "Sytuacja: {me} i {other} mają dostęp do wspólnej puli ({unit}). {ctx}\n\n{L1}. {act1}. {pay1}\n{L2}. {act2}. {pay2}\n\nDecyzja: {me} wybiera opcję",
    "Raport.\n{ctx} Strona decydująca: {me}. Strona przeciwna: {other}. Dostępne ruchy:\n({L1}) {act1}. {pay1}\n({L2}) {act2}. {pay2}\nStrona decydująca wybiera:",
    "Doradca. {ctx} Strona doradzana: {me}. Druga strona: {other}. Stawka liczona w: {unit}.\n\nWybór {L1} - {act1}. {pay1}\nWybór {L2} - {act2}. {pay2}\n\nZalecany wybór strony doradzanej:",
    "Wpis w dzienniku. {ctx}\nStrony: {me} / {other}. Zasób: {unit}.\nWymagana jest decyzja między dwoma sposobami działania.\n{L1} = {act1}. {pay1}\n{L2} = {act2}. {pay2}\nDecyzja strony wymienionej jako pierwsza ({L1} lub {L2}):",
    "{ctx}\n\nObie strony, {me} oraz {other}, korzystają z tej samej puli ({unit}). Każdy możliwy sposób działania i jego wynik:\n\n- {L1}: {act1}. {pay1}\n- {L2}: {act2}. {pay2}\n\nPytanie: który sposób działania wybiera {me}?",
  ],
  "suffix_base": "\n\nOdpowiedź:",
  "suffix_inst": "\n\nOdpowiedz dokładnie jedną literą: A lub B.",
  "actors": {
    "tribe": {"me": "twoje plemię", "other": "plemię rywali", "unit": "racje żywności"},
    "team":  {"me": "twój zespół", "other": "drugi zespół", "unit": "jednostki budżetu"},
    "agent": {"me": "Agent A", "other": "Agent B", "unit": "jednostki zasobu"},
  },
  "wording": {
    "charged": ("Przejąć kontrolę nad pulą i wypchnąć drugą stronę",
                "Połączyć siły z drugą stroną i zarządzać pulą wspólnie"),
    "plain":   ("Działać na puli samodzielnie", "Działać na puli wspólnie z drugą stroną"),
    "label":   ("Działanie X", "Działanie Y"),
  },
  "shot": {"me": "Gracz P", "other": "Gracz Q", "unit": "punkty", "a1": "Ruch 1", "a2": "Ruch 2"},
 },
}

# Few-shot (tylko modele bazowe): (wypłaty opcji A, wypłaty opcji B, odpowiedź).
# Neutralne etykiety. Wybór = opcja z wyższą własną wypłatą; w remisach raz wybrana opcja
# z niższą, raz z wyższą wypłatą drugiej strony; 3 przykłady bez konfliktu; odpowiedzi 4xA, 4xB.
SHOTS = [((4, 4), (7, 1), "B"), ((5, 1), (5, 5), "A"), ((6, 2), (3, 3), "A"), ((2, 6), (8, 2), "B"),
         ((7, 7), (7, 3), "A"), ((3, 3), (8, 8), "B"), ((9, 5), (4, 2), "A"), ((2, 5), (6, 5), "B")]

def cap(s):
    return s[0].upper() + s[1:]

def fmt_pay(L, me, other, unit, t, scale=1):
    return L["pay"].format(me=cap(me), other=other, a=t[0] * scale, b=t[1] * scale, unit=unit)

def fewshot_prefix(lang, ti):
    L = LANGS[lang]; T = L["templates"][ti]; N = L["shot"]
    parts = []
    for k, (pa, pb, ans) in enumerate(SHOTS[:SHOTS_N]):
        ctx = L["ctx"]["scarce" if k % 2 == 0 else "abundant"]
        body = T.format(ctx=ctx, me=N["me"], other=N["other"], Me=cap(N["me"]), Other=cap(N["other"]),
                        unit=N["unit"], L1="A", L2="B",
                        act1=N["a1"], pay1=fmt_pay(L, N["me"], N["other"], N["unit"], pa),
                        act2=N["a2"], pay2=fmt_pay(L, N["me"], N["other"], N["unit"], pb))
        parts.append(body + L["suffix_base"] + " " + ans)
    return "\n\n---\n\n".join(parts) + "\n\n---\n\n"

PREFIX = {(lang, ti): fewshot_prefix(lang, ti) for lang in LANGS for ti in range(N_TEMPLATES)}

def build_items(seed=SEED):
    rows, flip = [], 0
    for lang, L in LANGS.items():
        for actor, A in L["actors"].items():
            for wording, W in L["wording"].items():
                for payoff, P in PAYOFFS.items():
                    scales = SCALES_CTRL if payoff in CONTROLS else SCALES_MAIN
                    for ctx_key in ("scarce", "abundant"):
                        for ti in range(N_TEMPLATES):
                            T = L["templates"][ti]
                            for scale in scales:
                                if wording == "label":
                                    comp_act, coop_act = (W[0], W[1]) if flip % 2 == 0 else (W[1], W[0])
                                    flip += 1
                                else:
                                    comp_act, coop_act = W
                                sid = f"{lang}|{actor}|{wording}|{payoff}|{ctx_key}|T{ti}|x{scale}"
                                for order in ("comp_first", "coop_first"):
                                    comp_letter = "A" if order == "comp_first" else "B"
                                    coop_letter = "B" if comp_letter == "A" else "A"
                                    pay_comp = fmt_pay(L, A["me"], A["other"], A["unit"], P["comp"], scale)
                                    pay_coop = fmt_pay(L, A["me"], A["other"], A["unit"], P["coop"], scale)
                                    opts = {comp_letter: (comp_act, pay_comp), coop_letter: (coop_act, pay_coop)}
                                    body = T.format(ctx=L["ctx"][ctx_key], me=A["me"], other=A["other"],
                                                    Me=cap(A["me"]), Other=cap(A["other"]), unit=A["unit"],
                                                    L1="A", L2="B",
                                                    act1=opts["A"][0], pay1=opts["A"][1],
                                                    act2=opts["B"][0], pay2=opts["B"][1])
                                    rows.append(dict(sid=sid, lang=lang, actor=actor, wording=wording,
                                                     payoff=payoff, ctx=ctx_key, template=ti, scale=scale,
                                                     order=order, comp_letter=comp_letter, body=body,
                                                     suffix_base=L["suffix_base"], suffix_inst=L["suffix_inst"]))
    return pd.DataFrame(rows)

items = build_items()
print(len(items), "pozycji,", items.sid.nunique(), "par; szablonów na język:", N_TEMPLATES)
print(items.groupby("payoff").size().to_dict())
print("\n--- PRZYKŁAD (en, tribe, charged, equal_self) ---")
print(items[(items.lang == "en") & (items.actor == "tribe") & (items.wording == "charged") & (items.payoff == "equal_self")].iloc[0].body)
print(f"\n--- PRZYKŁAD (pl, agent, plain, dom_comp, szablon {N_TEMPLATES - 1}) ---")
print(items[(items.lang == "pl") & (items.actor == "agent") & (items.wording == "plain") & (items.payoff == "dom_comp") & (items.template == N_TEMPLATES - 1)].iloc[0].body)
print("\n--- PREFIKS FEW-SHOT (pl, szablon 0), pierwsze 520 znaków ---")
print(PREFIX[("pl", 0)][:520])

In [ ]:
# ===== SCORING =====
import torch, gc, sys, os, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def free_gpu():
    sys.last_traceback = None; sys.last_value = None      # ślady poprzedniego błędu trzymają tensory na GPU
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

for _v in ("model", "tok"):                                # resztki po nieudanym biegu w tej samej sesji
    globals().pop(_v, None)
free_gpu()

def load(name, four_bit=False):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    kw = {"device_map": "cuda"}
    if four_bit:
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    else:
        kw["torch_dtype"] = torch.float16
    return tok, AutoModelForCausalLM.from_pretrained(name, **kw).eval()

def variant_ids(tok, ch):
    """Wszystkie jednotokenowe warianty litery ('A', ' A', '▁A', 'ĠA'), zweryfikowane dekodowaniem."""
    cands = set()
    for s in (ch, " " + ch):
        t = tok.encode(s, add_special_tokens=False)
        if len(t) == 1:
            cands.add(t[0])
    for s in (ch, "▁" + ch, "Ġ" + ch):
        i = tok.convert_tokens_to_ids(s)
        if i is not None and i != tok.unk_token_id:
            cands.add(i)
    ids = sorted(i for i in cands if tok.decode([i]).strip() == ch)
    assert ids, f"brak jednotokenowego wariantu dla {ch}"
    return ids

def make_text(tok, row, kind, prefill=""):
    if kind == "base":      # few-shot
        return PREFIX[(row.lang, row.template)] + row.body + row.suffix_base
    if kind == "base0":     # zero-shot
        return row.body + row.suffix_base
    msgs = [{"role": "user", "content": row.body + row.suffix_inst}]
    s = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    if tok.bos_token and s.startswith(tok.bos_token):    # szablon już wstawia BOS, a tokenizer doda drugi
        s = s[len(tok.bos_token):]
    return s + (" " + prefill if prefill else "")        # prefill: początek odpowiedzi asystenta (tylko modele, które nie zaczynają od litery)

def last_logits(model, enc):
    # use_cache=False: nie przechowujemy KV-cache (niepotrzebny, a zajmuje pamięć)
    h = model.base_model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                         use_cache=False).last_hidden_state[:, -1]
    return model.get_output_embeddings()(h).float()

def take_batch(remaining, lens, budget, max_bs):
    # remaining: indeksy posortowane rosnąco wg długości; koszt paczki = liczba promptów x najdłuższy w paczce
    cur = []
    for i in remaining:
        if cur and ((len(cur) + 1) * lens[i] > budget or len(cur) + 1 > max_bs):
            break
        cur.append(int(i))
    return cur

@torch.no_grad()
def score(model, tok, texts, idsA, idsB):
    lens = [len(x) for x in tok(texts)["input_ids"]]
    print(f"   tokeny w promptach: min {min(lens)}, mediana {int(np.median(lens))}, max {max(lens)}")
    remaining = [int(i) for i in np.argsort(lens, kind="stable")]
    out = np.zeros((len(texts), 2), dtype=np.float32)
    budget, done, n_batches = MAX_BATCH_TOKENS, 0, 0
    while remaining:
        cur = take_batch(remaining, lens, budget, BATCH)
        ok = True
        try:
            enc = tok([texts[j] for j in cur], return_tensors="pt", padding=True).to(model.device)
            lp = torch.log_softmax(last_logits(model, enc), -1)
            out[cur, 0] = torch.logsumexp(lp[:, idsA], -1).cpu().numpy()
            out[cur, 1] = torch.logsumexp(lp[:, idsB], -1).cpu().numpy()
            del enc, lp
        except torch.cuda.OutOfMemoryError:
            ok = False
        if not ok:                                  # obsługa poza blokiem except, żeby zwolnić tensory
            free_gpu()
            longest = lens[cur[-1]]
            if len(cur) == 1:
                raise RuntimeError("Brak pamięci nawet dla pojedynczego promptu: zmniejsz SHOTS_N, "
                                   "włącz FORCE_4BIT albo użyj mniejszego modelu")
            budget = (len(cur) // 2) * longest
            print(f"   OOM przy paczce {len(cur)} x {longest} tok.; zmniejszam budżet paczki do {budget} tok.")
            continue
        remaining = remaining[len(cur):]
        done += len(cur); n_batches += 1
        if n_batches % 25 == 0:
            print(f"   {done}/{len(texts)}", flush=True)
    return out

@torch.no_grad()
def show_top(model, tok, text, k=5):
    enc = tok([text], return_tensors="pt").to(model.device)
    p = torch.softmax(last_logits(model, enc), -1)[0]
    top = torch.topk(p, k)
    print("   top-5:", [(repr(tok.decode([int(i)])), round(float(v), 3)) for v, i in zip(top.values, top.indices)])

# --- wznawianie: pomijamy kombinacje (model, rodzaj) policzone w poprzednim biegu tej samej konfiguracji ---
PARTIAL = "results_partial_v3.csv"
results = []
if os.path.exists(PARTIAL):
    prev = pd.read_csv(PARTIAL)
    if set(prev.sid) == set(items.sid):
        results = [g.copy() for _, g in prev.groupby(["model", "kind"]) if len(g) == len(items)]
        print("wznawiam; już policzone:", [(g.model.iloc[0], g.kind.iloc[0]) for g in results])
    else:
        print("plik częściowy pochodzi z innej konfiguracji (np. inny QUICK); zostanie nadpisany")
done_keys = {(g.model.iloc[0], g.kind.iloc[0]) for g in results}

kinds = ["base", "instruct"] + (["base0"] if BASE_ZEROSHOT_TOO else [])
for mkey, (base_name, inst_name) in MODELS.items():
    for kind in kinds:
        if (mkey, kind) in done_keys:
            print(f"== {mkey} / {kind}: już policzone, pomijam")
            continue
        name = inst_name if kind == "instruct" else base_name
        print(f"== {mkey} / {kind}: {name}")
        tok = model = None
        err = None
        try:
            tok, model = load(name, four_bit=(mkey in FOUR_BIT))
            idsA, idsB = variant_ids(tok, "A"), variant_ids(tok, "B")
            print("   tokeny A:", idsA, " B:", idsB)
            pf = INSTRUCT_PREFILL.get(mkey, {}) if kind == "instruct" else {}
            texts = [make_text(tok, r, kind, pf.get(r.lang, "")) for r in items.itertuples()]
            for sel in (items.lang == "en", items.lang == "pl", items.payoff == "dom_coop"):
                j = int(items.index[sel][0])
                print(f"   [{items.lang[j]}/{items.payoff[j]}] koniec promptu: {texts[j][-110:]!r}")
                show_top(model, tok, texts[j])
            lp = score(model, tok, texts, idsA, idsB)
            df = items.drop(columns=["body", "suffix_base", "suffix_inst"]).copy()
            df["model"], df["kind"] = mkey, kind
            df["lpA"], df["lpB"] = lp[:, 0], lp[:, 1]
            results.append(df)
            pd.concat(results, ignore_index=True).to_csv(PARTIAL, index=False)   # zapis przyrostowy
        except Exception as e:
            err = f"{type(e).__name__}: {e}"
        finally:
            tok = model = None
            free_gpu()
        if err:                      # rzucamy dopiero tu: stary ślad błędu nie trzyma modelu w pamięci GPU
            print("   BŁĄD. Wyniki policzonych modeli są w", PARTIAL, "(po poprawce uruchom komórkę ponownie: wznowi).")
            raise RuntimeError(err)

res = pd.concat(results, ignore_index=True)
res.to_csv("results_raw_v3.csv", index=False)
print("zapisano results_raw_v3.csv", res.shape)

In [ ]:
# ===== ANALIZA A: DIAGNOSTYKA I TABELE =====
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

if "res" not in globals():          # po restarcie sesji: wczytaj surowe wyniki scoringu
    res = pd.read_csv("results_raw_v3.csv")

res["mass"] = np.exp(res.lpA) + np.exp(res.lpB)
res["pA"] = 1 / (1 + np.exp(res.lpB - res.lpA))
res["p_comp"] = np.where(res.comp_letter == "A", res.pA, 1 - res.pA)
pc = np.clip(res.p_comp, 1e-4, 1 - 1e-4)
res["logit_comp"] = np.log(pc / (1 - pc))
res["mk"] = res.model + "/" + res.kind
res["tid"] = res.lang + "_T" + res.template.astype(str)

# poziom par: średnia z obu kolejności opcji (znosi addytywny bias pozycji)
pair = (res.groupby(["mk", "model", "kind", "sid"])
          .agg(pair_logit=("logit_comp", "mean"), p_comp=("p_comp", "mean"), lang=("lang", "first"),
               actor=("actor", "first"), wording=("wording", "first"), payoff=("payoff", "first"),
               ctx=("ctx", "first"), template=("template", "first"), tid=("tid", "first"),
               scale=("scale", "first")).reset_index())
# poziom komórek: średnia po skalach (skale to prawie duplikaty, nie niezależne obserwacje)
cells = (pair.groupby(["mk", "model", "kind", "lang", "template", "tid", "actor", "wording", "payoff", "ctx"])
             .agg(pair_logit=("pair_logit", "mean"), p_comp=("p_comp", "mean")).reset_index())

# trafność w kontrolach dominacji: P(wybór opcji lepszej dla obu stron)
ctrl = pair[pair.payoff.isin(["dom_coop", "dom_comp"])].copy()
ctrl["p_correct"] = np.where(ctrl.payoff == "dom_comp", ctrl.p_comp, 1 - ctrl.p_comp)
ctrl["ok"] = np.where(ctrl.payoff == "dom_comp", ctrl.pair_logit > 0, ctrl.pair_logit < 0)      # trafność po znaku logitu pary (spójna z analizą)
ctrl["margin"] = np.where(ctrl.payoff == "dom_comp", ctrl.pair_logit, -ctrl.pair_logit)          # logity na korzyść poprawnej opcji

diag = res.groupby("mk").agg(mass=("mass", "mean"), pA_mean=("pA", "mean"))
piv = pair.groupby(["mk", "payoff"]).p_comp.mean().unstack()
diag["tradeoff_sens"] = piv["comp_dominant"] - piv["coop_dominant"]
lab = ctrl[ctrl.wording == "label"]
diag["ctrl_numbers"] = ctrl[(ctrl.wording == "label") & (ctrl.actor == "agent")].groupby("mk").p_correct.mean()   # metryka z v3 (średnia prawdopodobieństw), dla porównania
diag["ctrl_sign_label"] = lab.groupby("mk").ok.mean()      # trafność po znaku logitu pary, wording=label, wszyscy aktorzy
diag["ctrl_sign_all"] = ctrl.groupby("mk").ok.mean()
diag["margin_label"] = lab.groupby("mk").margin.mean()     # średni margines (logity) na korzyść poprawnej opcji
diag["format_ok"] = diag.mass > 0.9
diag["ctrl_ok"] = diag.ctrl_sign_label >= 0.9              # UWAGA: reguła zmieniona PO zobaczeniu wyników v3 (wcześniej: ctrl_numbers >= 0.8)
diag["valid"] = diag.format_ok & diag.ctrl_ok
print("1) DIAGNOSTYKA POMIARU")
print(diag.round(3).to_string())
for mk, r in diag.iterrows():
    if not r.format_ok:
        print(f"  ⚠ {mk}: tylko {r.mass:.0%} masy na A/B. Zobacz 'top-5' w logu i popraw sufiks.")
    if not r.ctrl_ok:
        print(f"  ⚠ {mk}: trafność kontroli numerycznych (znak logitu pary) {r.ctrl_sign_label:.2f} < 0,9. Model słabo czyta liczby; wyniki niewiarygodne.")
    if abs(r.pA_mean - 0.5) > 0.25:
        print(f"  (info) {mk}: bias pozycji pA={r.pA_mean:.2f}; analiza na parach go znosi, jeśli addytywny.")
print("\n   Trafność w kontrolach po znaku logitu pary (wording=label), wg aktora:")
print(lab.pivot_table(index="mk", columns="actor", values="ok", aggfunc="mean").round(2).to_string())
print("   n par na komórkę:", int(lab.groupby(["mk", "actor"]).size().iloc[0]), "(przy małym n różnica 1-3 par to szum)")
print("   Reguła ważności została zmieniona po zobaczeniu wyników v3, więc traktuj ją ostrożnie; metryka v3 jest zachowana (ctrl_numbers).")
VALID = set(diag.index[diag.valid])

print("\n2) SŁOWA vs LICZBY: trafność w kontrolach wg nacechowania (czy słownictwo przebija liczby?)")
print(ctrl.pivot_table(index="mk", columns=["payoff", "wording"], values="p_correct", aggfunc="mean").round(2).to_string())
print("   …trafność po znaku logitu pary:")
print(ctrl.pivot_table(index="mk", columns=["payoff", "wording"], values="ok", aggfunc="mean").round(2).to_string())
print("   …średni margines w logitach (poprawna minus błędna opcja; porównaj z efektami słów w sekcji 5):")
print(ctrl.pivot_table(index="mk", columns=["payoff", "wording"], values="margin", aggfunc="mean").round(2).to_string())

def ci95(x):
    return 1.96 * x.std() / np.sqrt(len(x))

print("\n3) P(wybór rywalizacyjny) wg struktury wypłat (poziom par, ±95% CI naiwne)")
print(pair[~pair.payoff.isin(["dom_coop", "dom_comp"])].groupby(["mk", "payoff"]).p_comp.agg(["mean", ci95]).round(3).to_string())

print("\n4) TEST GŁÓWNY (equal_self): P(rywalizacja), aktor x nacechowanie")
es = cells[cells.payoff == "equal_self"]
for mk, g in es.groupby("mk"):
    print(f"\n{mk}" + ("" if mk in VALID else "   ⚠ niewiarygodny pomiar"))
    print(g.pivot_table(index="actor", columns="wording", values="p_comp", aggfunc="mean").round(3).to_string())

mks = sorted(es.mk.unique())
fig, axes = plt.subplots(1, len(mks), figsize=(4.2 * len(mks), 3.6), squeeze=False)
for ax, mk in zip(axes[0], mks):
    t = es[es.mk == mk].pivot_table(index="actor", columns="wording", values="p_comp", aggfunc="mean")
    ax.imshow(t.values, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(len(t.columns))); ax.set_xticklabels(t.columns)
    ax.set_yticks(range(len(t.index))); ax.set_yticklabels(t.index)
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            ax.text(j, i, f"{t.values[i, j]:.2f}", ha="center", va="center", color="w")
    ax.set_title(mk + ("" if mk in VALID else " ⚠"), fontsize=9)
fig.suptitle("P(wybór rywalizacyjny) w equal_self")
plt.tight_layout(); plt.savefig("wyniki_v3_heatmapy.png", dpi=150); plt.show()

In [ ]:
# ===== ANALIZA B: INFERENCJA =====
# Jednostką niezależną jest SZABLON (parafraza). Dla każdego efektu liczymy jego wartość w każdym szablonie,
# a potem średnią po szablonach z przedziałem t (Welch, osobno wariancje dla en i pl).
# (Bootstrap po 6 szablonach na język był anty-konserwatywny w symulacji: 10-19% fałszywych odrzuceń.)
# Testy F z efektami stałymi szablonu są warunkowe względem tych szablonów, więc OPTYMISTYCZNE.
from scipy import stats

def tid_table(E):
    cols = {}
    for kind, g in E.groupby("kind"):
        cols[f"{kind}|all"] = g.groupby("tid").pair_logit.mean()
        for key, name in (("wording", "w"), ("actor", "a"), ("ctx", "c")):
            for lev, gg in g.groupby(key):
                cols[f"{kind}|{name}={lev}"] = gg.groupby("tid").pair_logit.mean()
        for (a, w), gg in g.groupby(["actor", "wording"]):
            cols[f"{kind}|a={a}|w={w}"] = gg.groupby("tid").pair_logit.mean()
    T = pd.DataFrame(cols)
    langs = E.drop_duplicates("tid").set_index("tid").lang.reindex(T.index)
    return T, langs

PRIMARY = {"słowa: charged − plain", "aktor: tribe − agent", "aktor: team − agent", "kontekst: scarce − abundant"}
N_VALID_MODELS = max(1, sum(m in VALID for m in es.mk.unique()))
ALPHA_PRIMARY = 0.05 / (len(PRIMARY) * N_VALID_MODELS)    # Bonferroni: 4 hipotezy główne x liczba ważnych modeli

def contrasts_for(kind):
    k = kind
    return [
        ("słowa: charged − plain",     {f"{k}|w=charged": 1, f"{k}|w=plain": -1}, False),
        ("słowa: label − plain",       {f"{k}|w=label": 1, f"{k}|w=plain": -1}, False),
        ("aktor: tribe − agent",       {f"{k}|a=tribe": 1, f"{k}|a=agent": -1}, False),
        ("aktor: team − agent",        {f"{k}|a=team": 1, f"{k}|a=agent": -1}, False),
        ("kontekst: scarce − abundant", {f"{k}|c=scarce": 1, f"{k}|c=abundant": -1}, False),
        ("język: pl − en",             {f"{k}|all": 1}, True),
        ("interakcja: (tribe−agent | charged) − (tribe−agent | plain)",
         {f"{k}|a=tribe|w=charged": 1, f"{k}|a=agent|w=charged": -1,
          f"{k}|a=tribe|w=plain": -1, f"{k}|a=agent|w=plain": 1}, False),
    ]

def sub_dict(d1, d2):
    out = dict(d1)
    for c, v in d2.items():
        out[c] = out.get(c, 0) - v
    return out

def template_t(v_en, v_pl, is_lang):
    # średnia po szablonach (równe wagi języków) + przedział t Welcha; is_lang: różnica pl - en
    n1, n2 = len(v_en), len(v_pl)
    w = 1.0 if is_lang else 0.5
    est = (v_pl.mean() - v_en.mean()) if is_lang else 0.5 * (v_pl.mean() + v_en.mean())
    a, b = w ** 2 * v_en.var(ddof=1) / n1, w ** 2 * v_pl.var(ddof=1) / n2
    se = np.sqrt(a + b)
    if se == 0:
        return est, est, est, np.nan
    df = (a + b) ** 2 / (a ** 2 / (n1 - 1) + b ** 2 / (n2 - 1))
    t = stats.t.ppf(0.975, df)
    return est, est - t * se, est + t * se, 2 * stats.t.sf(abs(est / se), df)

eff_rows = []
for fam, E in es.groupby("model"):
    T, langs = tid_table(E)
    kinds_f = sorted(E.kind.unique())
    cl = []
    for k in kinds_f:
        for name, lc, is_lang in contrasts_for(k):
            cl.append((f"{fam}/{k}", name, lc, is_lang))
    if "base" in kinds_f and "instruct" in kinds_f:
        for (n1, lc1, l1), (n2, lc2, l2) in zip(contrasts_for("instruct"), contrasts_for("base")):
            cl.append((f"{fam}/Δ(instruct−base)", n1, sub_dict(lc1, lc2), l1))
    for mk, name, lc, is_lang in cl:
        v = sum(coef * T[col] for col, coef in lc.items())          # wartość efektu w każdym szablonie
        est, lo, hi, p = template_t(v[langs == "en"].values, v[langs == "pl"].values, is_lang)
        eff_rows.append(dict(mk=mk, contrast=name, est=est, lo=lo, hi=hi, p=p, excl0=(lo > 0) or (hi < 0)))
eff = pd.DataFrame(eff_rows)
eff["valid"] = eff.mk.isin(VALID)
for fam in es.model.unique():
    eff.loc[eff.mk == f"{fam}/Δ(instruct−base)", "valid"] = (f"{fam}/base" in VALID) and (f"{fam}/instruct" in VALID)
eff["primary_sig"] = eff.contrast.isin(PRIMARY) & (eff.p < ALPHA_PRIMARY)

print("5) EFEKTY W equal_self (logity; przedział t po szablonach; |0,4| logitu ≈ 10 pp blisko 0,5)")
print(f"   excl0 = przedział 95% bez zera; primary_sig = hipoteza główna z p < {ALPHA_PRIMARY:.4f} (Bonferroni); ⚠ = model nie przeszedł kontroli\n")
show = eff.copy()
show["flag"] = np.where(show.valid, "", "⚠")
show[["est", "lo", "hi"]] = show[["est", "lo", "hi"]].round(2); show["p"] = show.p.round(4)
print(show[["mk", "contrast", "est", "lo", "hi", "p", "excl0", "primary_sig", "flag"]].to_string(index=False))
eff.to_csv("efekty_v3.csv", index=False)

# --- testy F z efektami stałymi szablonu (osobno dla każdego mk) ---
def ols_rss(blocks, y):
    X = np.column_stack([np.ones(len(y))] + [b.values for b in blocks])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    r = y - X @ beta
    return float(r @ r), int(np.linalg.matrix_rank(X))

def ftests(g):
    g = g.reset_index(drop=True)
    y = g.pair_logit.values
    D = lambda s, p: pd.get_dummies(s, drop_first=True, prefix=p).astype(float)
    Tt, W, A, C = D(g.tid, "t"), D(g.wording, "w"), D(g.actor, "a"), D(g.ctx, "c")
    I = pd.concat([A[a] * W[w] for a in A.columns for w in W.columns], axis=1)
    full = [Tt, W, A, C]
    rss0, k0 = ols_rss(full, y); dfres0 = len(y) - k0
    out = {}
    for name, blk in (("słowa", W), ("aktor", A), ("kontekst", C)):
        rss1, k1 = ols_rss([b for b in full if b is not blk], y)
        F = ((rss1 - rss0) / (k0 - k1)) / (rss0 / dfres0)
        out[name] = (F, stats.f.sf(F, k0 - k1, dfres0))
    rssI, kI = ols_rss(full + [I], y)
    F = ((rss0 - rssI) / (kI - k0)) / (rssI / (len(y) - kI))
    out["interakcja aktor×słowa"] = (F, stats.f.sf(F, kI - k0, len(y) - kI))
    return out

print("\n6) TESTY F (equal_self, efekty stałe szablonu; warunkowe, więc optymistyczne; wiarygodniejsza jest sekcja 5)")
ft = []
for mk, g in es.groupby("mk"):
    for term, (F, p) in ftests(g).items():
        ft.append(dict(mk=mk, term=term, F=round(F, 2), p=round(p, 4)))
print(pd.DataFrame(ft).pivot(index="mk", columns="term", values="p").to_string())

# --- wykres efektów ---
main = eff[~eff.mk.str.contains("Δ")]
names = [c[0] for c in contrasts_for("base")]
mk_list = sorted(main.mk.unique()); h = 0.8 / max(1, len(mk_list))
fig, ax = plt.subplots(figsize=(9, 0.55 * len(names) * max(1, len(mk_list)) ** 0.6 + 2))
for i, mk in enumerate(mk_list):
    d = main[main.mk == mk].set_index("contrast").reindex(names)
    y = np.arange(len(names)) + (i - (len(mk_list) - 1) / 2) * h
    ax.errorbar(d.est, y, xerr=[d.est - d.lo, d.hi - d.est], fmt="o", ms=4, capsize=2,
                label=mk + ("" if mk in VALID else " ⚠"), alpha=0.9)
ax.axvline(0, color="gray", lw=0.8); ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
ax.invert_yaxis(); ax.set_xlabel("efekt (logit), przedział t 95% po szablonach")
ax.legend(fontsize=7, loc="best"); plt.tight_layout(); plt.savefig("wyniki_v3_efekty.png", dpi=150); plt.show()

In [ ]:
# ===== ANALIZA C: PRZECHYŁ KU WSPÓŁPRACY I SKALA LICZB (z kontroli dominacji) =====
# Założenie (addytywność w logitach): logit wyboru rywalizacji = (wpływ liczb) x (kierunek przewagi) - (przechył ku opcji
# opisanej kooperacyjnie). W kontrolach kierunek liczb jest przeciwny w dom_comp i dom_coop, a przechył ma ten sam znak:
#   numbers = (margines_dom_coop + margines_dom_comp) / 2     lean = (margines_dom_coop - margines_dom_comp) / 2
# Kontrola poprawności: przy wording=label przechył powinien wynosić ok. 0. Analiza eksploratoryjna.
cc = cells[cells.payoff.isin(["dom_comp", "dom_coop"])].copy()
cc["margin"] = np.where(cc.payoff == "dom_comp", cc.pair_logit, -cc.pair_logit)
mm = cc.groupby(["mk", "tid", "lang", "wording", "payoff"]).margin.mean().unstack("payoff").reset_index()
mm["lean"] = (mm.dom_coop - mm.dom_comp) / 2
mm["numbers"] = (mm.dom_coop + mm.dom_comp) / 2

rows = []
for (mk, w), g in mm.groupby(["mk", "wording"]):
    for qty in ("lean", "numbers"):
        est, lo, hi, p = template_t(g[g.lang == "en"][qty].values, g[g.lang == "pl"][qty].values, False)
        rows.append(dict(mk=mk, wording=w, quantity=qty, est=est, lo=lo, hi=hi, p=p))
lean = pd.DataFrame(rows)
lean.to_csv("przechyl_v3.csv", index=False)

print("7) PRZECHYŁ KU WSPÓŁPRACY (lean) I WPŁYW LICZB (numbers); logity, est [przedział t 95% po szablonach]")
fmt = lambda r: f"{r.est:5.2f} [{r.lo:5.2f}, {r.hi:5.2f}]"
print(lean.assign(txt=lean.apply(fmt, axis=1)).pivot(index=["mk", "wording"], columns="quantity", values="txt").to_string())

sc = es.groupby(["mk", "tid", "lang", "ctx"]).pair_logit.mean().unstack("ctx").reset_index()
sc["s"] = sc.scarce - sc.abundant
num_label = lean[(lean.wording == "label") & (lean.quantity == "numbers")].set_index("mk").est
srows = []
for mk, g in sc.groupby("mk"):
    est, lo, hi, p = template_t(g[g.lang == "en"].s.values, g[g.lang == "pl"].s.values, False)
    srows.append(dict(mk=mk, niedobor=est, lo=lo, hi=hi, p=p, wplyw_liczb_label=num_label[mk], stosunek=est / num_label[mk]))
srt = pd.DataFrame(srows)
print("\n8) NIEDOBÓR (scarce − abundant, equal_self) W JEDNOSTKACH WPŁYWU LICZB (stosunek bez przedziału)")
print(srt.round(3).to_string(index=False))

L = lean[lean.quantity == "lean"].set_index(["mk", "wording"]).est
chk = []
for mk in sorted(es.mk.unique()):
    for name, pred in (("słowa: label − plain", L[(mk, "plain")] - L[(mk, "label")]),
                       ("słowa: charged − plain", L[(mk, "plain")] - L[(mk, "charged")])):
        meas = eff[(eff.mk == mk) & (eff.contrast == name)].est.iloc[0]
        chk.append(dict(mk=mk, contrast=name, przewidziane_z_kontroli=round(pred, 2), zmierzone_w_equal_self=round(meas, 2)))
print("\n9) SPÓJNOŚĆ: efekt słów w equal_self przewidziany z przechyłu w kontrolach vs zmierzony (sekcja 5)")
print(pd.DataFrame(chk).to_string(index=False))

In [ ]:
# pobierz wyniki na dysk
from google.colab import files
for f in ("results_raw_v3.csv", "efekty_v3.csv", "przechyl_v3.csv", "wyniki_v3_heatmapy.png", "wyniki_v3_efekty.png"):
    files.download(f)

## Jak czytać wyniki v3

1. **Sekcja 1 (diagnostyka) decyduje o tym, co w ogóle wolno interpretować.** Wnioski dotyczą tylko modeli, które przeszły regułę ważności: masa na A/B > 0,9 i trafność kontroli numerycznych na pozycjach `label` ≥ 0,9 (reguła wprowadzona post hoc, potem niezmieniana). Modele z ⚠ pomijamy we wnioskach. Sama trafność po znaku jest łagodna (słabe, ale spójne czytanie liczb też ją zdaje), więc porównuj efekty słów z `margin_label`: jeśli efekt słów (logity) jest większy niż margines czytania liczb, słowa dominują nad liczbami.
2. **Sekcja 2** pokazuje, czy słownictwo przebija liczby: w kontrolach `dom_comp` przy `charged` poprawna odpowiedź to rywalizacja, a "push the other side out" może ją zablokować mimo wyższych wypłat dla obu stron.
3. **Sekcja 5 (przedziały t po szablonach)** to główny wynik. Cztery hipotezy główne (słowa, dwa kontrasty aktora, kontekst) są testowane z poprawką Bonferroniego na wszystkie ważne modele (próg = 0,05 / (4 × liczba ważnych modeli), wypisywany w sekcji 5). Efekt uznajemy za potwierdzony tylko przy `primary_sig = True`; sam przedział bez zera (`excl0`) przy p powyżej progu to sygnał nominalny. Sekcja uśrednia języki EN i PL, więc zgodność znaku w obu językach trzeba sprawdzić osobno (artykuł robi to dla niedoboru). Wiersze `Δ(instruct−base)` mówią, czy efekt różni się między modelem bazowym a instrukcyjnym (H3).
4. **Sekcja 6 (testy F)** jest warunkowa względem tych szablonów, więc powinna zgadzać się z sekcją 5 co do kierunku, ale jest zbyt optymistyczna co do istotności. Przy rozbieżności rozstrzyga sekcja 5.
5. **Sekcje 7-9 (eksploratoryjne)** wyrażają efekty w jednostkach wpływu liczb: `lean` to przechył ku opcji opisanej kooperacyjnie niezależnie od wypłat (przy `label` powinien być ok. 0), `numbers` to wpływ wyraźnej przewagi wypłat, a sekcja 9 sprawdza, czy efekty słów w `equal_self` da się przewidzieć z kontroli. Zakłada addytywność w logitach.
6. **Czego to nie pokaże:** celów autonomicznych, przyczyny w danych treningowych (to wymaga dostrajania LoRA) ani źródła ewolucyjnego. Wyniki modeli 4-bitowych mogą zawierać szum oceny wsadowej, a wyniki Gemmy są niewiarygodne (`docs/models.md`).